# 01 · The Bad

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/splevine/clustering-good-bad-beautiful/blob/main/notebooks/01_the_bad.ipynb)

**The failure modes that bite every clustering project.**

Clustering promises structure discovery. In practice you get back... *something*. The hard question is whether that something is real — and most of the time, the algorithm won't tell you.

This notebook walks through four failure modes on a concrete dataset: the **top 5,000 movies** by TMDB vote count. K-means appears as a demonstrator in a few of them, but it's not the villain — the failure modes are.

| # | Failure mode | What it looks like |
| - | --- | --- |
| 1 | Clusters from nothing | The algorithm returns clean-looking clusters on data with no real structure. |
| 2 | Instability across runs | Same data, different seed or init — different story. |
| 3 | Raw high-dim embeddings | Distances compress; everything looks equidistant from everything else. |
| 4 | Parameter choices *become* the story | Small knob changes flip cluster counts and interpretations. |

## Setup

In [ ]:
# Colab: uncomment to install
# !pip install -q sentence-transformers scikit-learn umap-learn pandas pyarrow matplotlib

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(0)

## Load movies and embed overviews

`all-MiniLM-L6-v2` is a 384-D sentence encoder — fast, solid default for short text like plot overviews. We cache the embeddings so re-running is instant.

In [ ]:
movies = pd.read_parquet("../data/movies.parquet")
movies = movies[movies["overview"].str.len() > 30].reset_index(drop=True)
overviews = movies["overview"].tolist()
print(f"{len(overviews):,} movies with non-trivial overviews")

In [ ]:
from sentence_transformers import SentenceTransformer

EMB_DIR = Path("../embeddings")
EMB_DIR.mkdir(exist_ok=True)
real_path = EMB_DIR / "overviews_minilm.npy"

if real_path.exists():
    X_real = np.load(real_path)
    print(f"loaded cached embeddings: {X_real.shape}")
else:
    model = SentenceTransformer("all-MiniLM-L6-v2")
    X_real = model.encode(overviews, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    np.save(real_path, X_real)
    print(f"computed and cached: {X_real.shape}")

## Failure mode 1 — Clusters from nothing

Clustering algorithms don't validate that clusters exist. They return a partition regardless. To make this visceral: generate random Gaussian vectors with the same shape as our real embeddings, then run k-means. Watch it happily return clean-looking clusters.

If "the algorithm gave me 8 groups" is your only evidence for structure, you have no evidence.

In [ ]:
from sklearn.cluster import KMeans
from umap import UMAP

K = 8

# Structureless data: same shape, same L2 normalization, pure noise.
X_noise = rng.standard_normal(X_real.shape).astype(np.float32)
X_noise /= np.linalg.norm(X_noise, axis=1, keepdims=True)

km_noise = KMeans(n_clusters=K, random_state=0, n_init=10).fit(X_noise)
km_real = KMeans(n_clusters=K, random_state=0, n_init=10).fit(X_real)

print("cluster sizes on pure noise:", np.bincount(km_noise.labels_).tolist())
print("cluster sizes on real data: ", np.bincount(km_real.labels_).tolist())

In [ ]:
# Visualize with UMAP so the difference is obvious.
umap_2d = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=0)
noise_2d = umap_2d.fit_transform(X_noise)
real_2d = umap_2d.fit_transform(X_real)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(noise_2d[:, 0], noise_2d[:, 1], c=km_noise.labels_, cmap="tab10", s=3, alpha=0.6)
axes[0].set_title(f"KMeans(k={K}) on Gaussian noise\n(no real structure, but clusters anyway)")
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[1].scatter(real_2d[:, 0], real_2d[:, 1], c=km_real.labels_, cmap="tab10", s=3, alpha=0.6)
axes[1].set_title(f"KMeans(k={K}) on real movie embeddings")
axes[1].set_xticks([]); axes[1].set_yticks([])
plt.tight_layout(); plt.show()

On the left, the clusters are a fiction — the data is isotropic Gaussian noise. On the right, the blobs have internal structure (we'll see this more clearly in `02_the_good`).

**Takeaway:** always test whether your data has clustering tendency *before* interpreting clusters.

## Failure mode 2 — Instability across runs

Real clusters should be stable under perturbation. To test this, fit k-means three times with different seeds (and `n_init=1` so each run gets one init — realistic for a quick pipeline). Then compute pairwise adjusted Rand index (ARI). ARI of 1 means identical clusterings; 0 means no better than random agreement.

In [ ]:
from sklearn.metrics import adjusted_rand_score

def cluster_several(X, k=K, seeds=(0, 1, 2, 3, 4), n_init=1):
    return {s: KMeans(n_clusters=k, random_state=s, n_init=n_init).fit_predict(X) for s in seeds}

def pairwise_ari(labels_by_seed):
    seeds = list(labels_by_seed)
    m = np.eye(len(seeds))
    for i, a in enumerate(seeds):
        for j, b in enumerate(seeds):
            if i < j:
                m[i, j] = m[j, i] = adjusted_rand_score(labels_by_seed[a], labels_by_seed[b])
    return pd.DataFrame(m, index=[f"s={s}" for s in seeds], columns=[f"s={s}" for s in seeds])

ari_raw = pairwise_ari(cluster_several(X_real, n_init=1))
print("Pairwise ARI, KMeans on raw 384-D embeddings (n_init=1):")
print(ari_raw.round(3))
print(f"\nMean off-diagonal ARI: {(ari_raw.values.sum() - np.trace(ari_raw.values)) / (ari_raw.size - len(ari_raw)):.3f}")

**Read:** clusters in raw high-D space shift substantially just by changing the seed. If a downstream decision depends on cluster #3 being "romantic comedies," that identity isn't stable.

(We'll see in `02_the_good` that UMAP-reducing *before* clustering makes this much more stable.)

## Failure mode 3 — Raw high-dimensional embeddings

At 384 dimensions, pairwise distances compress: nearest and farthest neighbors sit closer together than intuition suggests. Density-based clustering struggles because "near" and "far" become hard to tell apart.

We'll look at one summary: for each point, how far is its nearest neighbor compared to its farthest? If the ratio is close to 1, distances are compressed.

In [ ]:
from sklearn.metrics import pairwise_distances

sample = rng.choice(len(X_real), 1500, replace=False)
Xs = X_real[sample]

reducer = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric="cosine", random_state=0)
Xs_reduced = reducer.fit_transform(Xs)

def nearest_farthest_ratio(X, metric):
    d = pairwise_distances(X, metric=metric)
    np.fill_diagonal(d, np.inf)
    nearest = d.min(axis=1)
    np.fill_diagonal(d, -np.inf)
    farthest = d.max(axis=1)
    return nearest, farthest

n_raw, f_raw = nearest_farthest_ratio(Xs, "cosine")
n_red, f_red = nearest_farthest_ratio(Xs_reduced, "euclidean")

print(f"raw 384-D (cosine):  nearest/farthest mean ratio = {(n_raw/f_raw).mean():.3f}")
print(f"UMAP 5-D (euclid):   nearest/farthest mean ratio = {(n_red/f_red).mean():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist([n_raw, f_raw], bins=40, label=["nearest", "farthest"], edgecolor="black")
axes[0].set_title("Raw 384-D: nearest & farthest distances overlap")
axes[0].set_xlabel("cosine distance"); axes[0].legend()
axes[1].hist([n_red, f_red], bins=40, label=["nearest", "farthest"], edgecolor="black")
axes[1].set_title("UMAP 5-D: nearest & farthest are clearly separated")
axes[1].set_xlabel("euclidean distance"); axes[1].legend()
plt.tight_layout(); plt.show()

**Read:** in the raw space the two histograms overlap — "nearest" and "farthest" are barely distinguishable. Any density-based clustering in that space is working with degraded signal. After UMAP, the histograms separate, which is why clustering works better on reduced embeddings.

## Failure mode 4 — Parameter choices become the story

Pick a popular movie and ask: *who are its cluster-mates?* Then sweep `k` and watch the neighbors flip. If the story changes when you turn a knob, the story isn't about the data.

In [ ]:
target_title = "Inception"
target_idx = movies.index[movies["title"] == target_title][0]

def cluster_mates(X, k, target_idx, n=5):
    labels = KMeans(n_clusters=k, random_state=0, n_init=10).fit_predict(X)
    same = np.where(labels == labels[target_idx])[0]
    same = same[same != target_idx]
    # Pick the n most popular cluster-mates so the output is recognizable.
    ranked = movies.iloc[same].sort_values("vote_count", ascending=False)
    return ranked["title"].head(n).tolist(), len(same) + 1

print(f"Cluster-mates of '{target_title}' as k varies (top 5 by popularity):\n")
for k in [5, 10, 20, 40, 80]:
    mates, size = cluster_mates(X_real, k, target_idx)
    print(f"  k={k:3d} (cluster size {size:4d}): {', '.join(mates)}")

**Read:** at `k=5`, *Inception* sits with half the catalog. At `k=80`, it's pinned to a tight neighborhood of like-minded sci-fi thrillers. Neither is wrong — but the narrative you'd tell an exec about "Inception's cluster" is completely different at each setting.

Defensible clustering requires a **sensitivity sweep**, not a single "best" k. Better yet: use a method that infers cluster count from the data (next notebook).

## What "The Good" does differently

- **Tests for clustering tendency** before interpreting clusters.
- **Reduces dimensionality first** so geometry is interpretable and distances are meaningful.
- **Uses a density-based clusterer** that can say *"this movie isn't in any cluster."*
- **Treats parameters as ranges to sweep**, not oracles.
- **Labels clusters with interpretable representations** so humans can sanity-check.

Next: [`02_the_good.ipynb`](02_the_good.ipynb).